# 🫀 PULSE ECG — Training from Google Drive
**Runtime → Change runtime type → GPU (T4)** before running!

This version of the notebook expects that you have uploaded the **ENTIRE project**, including the `data/raw` folder, directly into your Google Drive.

---
## STEP 1: Mount Google Drive & Go to Project Folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ============================================================
# 👇 CHANGE THIS to your exact project path in Google Drive
# ============================================================
PROJECT_DIR = '/content/drive/MyDrive/pulse_ecg_model/pulse_ecg_model'

os.chdir(PROJECT_DIR)
print('\n✅ Current Directory:', os.getcwd())
print('📁 Files in project:', os.listdir('.'))

# Create required output folders (if they don't exist yet)
for d in ['data/processed', 'models/tfjs_model', 'logs', 'evaluation', 'checkpoints']:
    os.makedirs(d, exist_ok=True)

---
## STEP 2: Install Dependencies

In [ ]:
!pip install -q wfdb scipy scikit-learn matplotlib seaborn tqdm PyYAML h5py pandas tensorboard
print('\n✅ Dependencies installed')

---
## STEP 3: Verify GPU

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {gpus}')

if gpus:
    !nvidia-smi
    print('\n✅ GPU is ready!')
else:
    print('\n❌ No GPU detected! Go to: Runtime → Change runtime type → GPU (T4)')

---
## STEP 4: Train the Model 🚀
We are passing `--skip-download` because your raw datasets are already in `data/raw` on your Google Drive.

In [ ]:
!python train.py --skip-download

---
## STEP 5: View Training Results

In [ ]:
import json, glob, os
from IPython.display import Image, display

for ef in sorted(glob.glob('evaluation/*.json')):
    print(f'\n{"="*50}')
    print(f'📊 {os.path.basename(ef)}')
    with open(ef) as f:
        data = json.load(f)
        for k, v in data.items():
            if isinstance(v, float):
                print(f'  {k}: {v:.4f}')

for pf in sorted(glob.glob('evaluation/*.png')):
    print(f'\n📈 {os.path.basename(pf)}')
    display(Image(filename=pf, width=600))

---
## STEP 6: Convert to TensorFlow.js

In [ ]:
!pip install -q tensorflowjs
!python convert_to_tfjs.py --metadata

tfjs_dir = 'models/tfjs_model'
if os.path.exists(tfjs_dir):
    print(f'\n✅ TF.js model files exported successfully to {tfjs_dir}')
else:
    print('\n❌ TF.js conversion may have failed')